In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBClassifier, XGBRegressor

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

In [2]:
# Load dataset
df = pd.read_csv("../dataset/student-mat.csv", sep=";")

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (395, 33)

First 5 rows:


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [3]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

print("\nG3 statistics:")
print(df["G3"].describe())

Missing values: 0
Duplicate rows: 0

Data types:
school        object
sex           object
age            int64
address       object
famsize       object
Pstatus       object
Medu           int64
Fedu           int64
Mjob          object
Fjob          object
reason        object
guardian      object
traveltime     int64
studytime      int64
failures       int64
schoolsup     object
famsup        object
paid          object
activities    object
nursery       object
higher        object
internet      object
romantic      object
famrel         int64
freetime       int64
goout          int64
Dalc           int64
Walc           int64
health         int64
absences       int64
G1             int64
G2             int64
G3             int64
dtype: object

G3 statistics:
count    395.000000
mean      10.415190
std        4.581443
min        0.000000
25%        8.000000
50%       11.000000
75%       14.000000
max       20.000000
Name: G3, dtype: float64


In [4]:
# Create classification target
df["result"] = (df["G3"] >= 10).astype(int)

print("Classification target distribution:")
print(df["result"].value_counts())

print("\nPercentage distribution:")
print(df["result"].value_counts(normalize=True) * 100)

Classification target distribution:
result
1    265
0    130
Name: count, dtype: int64

Percentage distribution:
result
1    67.088608
0    32.911392
Name: proportion, dtype: float64


In [5]:
# IMPORTANT:
# G3 is the regression target.
# result is the classification target.
# Neither can be used as an input feature.

X = df.drop(["G3", "result"], axis=1)

y_classification = df["result"]
y_regression = df["G3"]

print("X shape:", X.shape)

print("\nTarget-like columns remaining in X:")
print([
    col for col in X.columns
    if "G3" in str(col).upper()
    or "RESULT" in str(col).upper()
])

print("\nFeatures:")
print(X.columns.tolist())

X shape: (395, 32)

Target-like columns remaining in X:
[]

Features:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2']


In [6]:
print("G1 in features:", "G1" in X.columns)
print("G2 in features:", "G2" in X.columns)
print("G3 in features:", "G3" in X.columns)
print("result in features:", "result" in X.columns)

G1 in features: True
G2 in features: True
G3 in features: False
result in features: False


In [7]:
X_train, X_test, y_cls_train, y_cls_test, y_reg_train, y_reg_test = train_test_split(
    X,
    y_classification,
    y_regression,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_classification
)

print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])

print("\nTraining feature shape:", X_train.shape)
print("Testing feature shape :", X_test.shape)

print("\nClassification training distribution:")
print(y_cls_train.value_counts())

print("\nClassification testing distribution:")
print(y_cls_test.value_counts())

Training samples: 316
Testing samples : 79

Training feature shape: (316, 32)
Testing feature shape : (79, 32)

Classification training distribution:
result
1    212
0    104
Name: count, dtype: int64

Classification testing distribution:
result
1    53
0    26
Name: count, dtype: int64


In [8]:
categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Number of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Number of numerical features: 15
Number of categorical features: 17

Numerical features:
['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2']

Categorical features:
['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']


In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [10]:
classification_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        n_jobs=-1
    )
}

In [11]:
classification_results = []
classification_pipelines = {}

for name, model in classification_models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_cls_train)

    y_pred = pipeline.predict(X_test)

    accuracy = accuracy_score(y_cls_test, y_pred)
    precision = precision_score(
        y_cls_test,
        y_pred,
        zero_division=0
    )
    recall = recall_score(
        y_cls_test,
        y_pred,
        zero_division=0
    )
    f1 = f1_score(
        y_cls_test,
        y_pred,
        zero_division=0
    )

    classification_results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

    classification_pipelines[name] = pipeline

classification_results_df = pd.DataFrame(
    classification_results
)

print("Classification Results")
print("======================")

display(
    classification_results_df.style.format({
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1 Score": "{:.4f}"
    })
)

Classification Results


,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.8734,0.9574,0.8491,0.9000
1,Decision Tree,0.8608,0.9773,0.8113,0.8866
2,Random Forest,0.8734,0.9574,0.8491,0.9000
3,XGBoost,0.8987,0.9787,0.8679,0.9200


In [12]:
classification_percentage = classification_results_df.copy()

for column in ["Accuracy", "Precision", "Recall", "F1 Score"]:
    classification_percentage[column] *= 100

display(
    classification_percentage.style.format({
        "Accuracy": "{:.2f}%",
        "Precision": "{:.2f}%",
        "Recall": "{:.2f}%",
        "F1 Score": "{:.2f}%"
    })
)

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,87.34%,95.74%,84.91%,90.00%
1,Decision Tree,86.08%,97.73%,81.13%,88.66%
2,Random Forest,87.34%,95.74%,84.91%,90.00%
3,XGBoost,89.87%,97.87%,86.79%,92.00%


In [13]:
best_classification_row = classification_results_df.loc[
    classification_results_df["Accuracy"].idxmax()
]

best_classification_name = best_classification_row["Model"]

print("Best Classification Model:")
print(best_classification_name)

print("\nBest Accuracy:")
print(
    f"{best_classification_row['Accuracy'] * 100:.2f}%"
)

print("\nBest F1 Score:")
print(
    f"{best_classification_row['F1 Score'] * 100:.2f}%"
)

Best Classification Model:
XGBoost

Best Accuracy:
89.87%

Best F1 Score:
92.00%


In [14]:
best_classification_pipeline = classification_pipelines[
    best_classification_name
]

best_cls_pred = best_classification_pipeline.predict(X_test)

cm = confusion_matrix(
    y_cls_test,
    best_cls_pred
)

print("Confusion Matrix")
print(cm)

Confusion Matrix
[[25  1]
 [ 7 46]]


In [15]:
from sklearn.model_selection import cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

best_cv_scores = cross_val_score(
    best_classification_pipeline,
    X,
    y_classification,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

print("5-Fold Cross-Validation Accuracy:")
print(best_cv_scores)

print("\nMean CV Accuracy:")
print(f"{best_cv_scores.mean() * 100:.2f}%")

print("Standard Deviation:")
print(f"{best_cv_scores.std() * 100:.2f}%")

5-Fold Cross-Validation Accuracy:
[0.91139241 0.88607595 0.89873418 0.88607595 0.91139241]

Mean CV Accuracy:
89.87%
Standard Deviation:
1.13%


In [16]:
regression_models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        random_state=RANDOM_STATE
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "XGBoost": XGBRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        objective="reg:squarederror",
        n_jobs=-1
    )
}

In [17]:
regression_results = []
regression_pipelines = {}

for name, model in regression_models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_reg_train)

    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(
        y_reg_test,
        y_pred
    )

    mse = mean_squared_error(
        y_reg_test,
        y_pred
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_reg_test,
        y_pred
    )

    regression_results.append({
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

    regression_pipelines[name] = pipeline

regression_results_df = pd.DataFrame(
    regression_results
)

display(
    regression_results_df.style.format({
        "MAE": "{:.4f}",
        "MSE": "{:.4f}",
        "RMSE": "{:.4f}",
        "R2": "{:.4f}"
    })
)

,Model,MAE,MSE,RMSE,R2
0,Linear Regression,1.5445,4.5943,2.1434,0.7712
1,Decision Tree,1.1519,5.3544,2.3140,0.7334
2,Random Forest,1.0692,2.6885,1.6397,0.8661
3,XGBoost,1.1275,2.9834,1.7273,0.8515


In [18]:
best_regression_row = regression_results_df.loc[
    regression_results_df["R2"].idxmax()
]

best_regression_name = best_regression_row["Model"]

print("Best Regression Model:")
print(best_regression_name)

print(
    f"\nR² = {best_regression_row['R2']:.4f}"
)

Best Regression Model:
Random Forest

R² = 0.8661


In [19]:
xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            XGBClassifier(
                random_state=RANDOM_STATE,
                eval_metric="logloss",
                n_jobs=-1
            )
        )
    ]
)

xgb_param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [2, 3, 4],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.8, 1.0]
}

xgb_grid = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=xgb_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(
    X_train,
    y_cls_train
)

print("Best Parameters:")
print(xgb_grid.best_params_)

print("\nBest Cross-Validation Accuracy:")
print(
    f"{xgb_grid.best_score_ * 100:.2f}%"
)

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best Parameters:
{'model__learning_rate': 0.05, 'model__max_depth': 2, 'model__n_estimators': 100, 'model__subsample': 0.8}

Best Cross-Validation Accuracy:
93.03%


In [21]:
tuned_xgb = xgb_grid.best_estimator_

y_pred_tuned_xgb = tuned_xgb.predict(X_test)

tuned_accuracy = accuracy_score(
    y_cls_test,
    y_pred_tuned_xgb
)

tuned_precision = precision_score(
    y_cls_test,
    y_pred_tuned_xgb,
    zero_division=0
)

tuned_recall = recall_score(
    y_cls_test,
    y_pred_tuned_xgb,
    zero_division=0
)

tuned_f1 = f1_score(
    y_cls_test,
    y_pred_tuned_xgb,
    zero_division=0
)

print("Tuned XGBoost")
print("=============")

print(f"Accuracy : {tuned_accuracy * 100:.2f}%")
print(f"Precision: {tuned_precision * 100:.2f}%")
print(f"Recall   : {tuned_recall * 100:.2f}%")
print(f"F1 Score : {tuned_f1 * 100:.2f}%")

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_cls_test,
    y_pred_tuned_xgb
))

Tuned XGBoost
Accuracy : 87.34%
Precision: 95.74%
Recall   : 84.91%
F1 Score : 90.00%

Confusion Matrix:
[[24  2]
 [ 8 45]]


In [22]:
tuned_result = pd.DataFrame([{
    "Model": "Tuned XGBoost",
    "Accuracy": tuned_accuracy,
    "Precision": tuned_precision,
    "Recall": tuned_recall,
    "F1 Score": tuned_f1
}])

all_classification_results = pd.concat(
    [
        classification_results_df,
        tuned_result
    ],
    ignore_index=True
)

display(
    all_classification_results.style.format({
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1 Score": "{:.4f}"
    })
)

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.8734,0.9574,0.8491,0.9000
1,Decision Tree,0.8608,0.9773,0.8113,0.8866
2,Random Forest,0.8734,0.9574,0.8491,0.9000
3,XGBoost,0.8987,0.9787,0.8679,0.9200
4,Tuned XGBoost,0.8734,0.9574,0.8491,0.9000


In [23]:
best_final_row = all_classification_results.loc[
    all_classification_results["Accuracy"].idxmax()
]

print("Final selected model:")
print(best_final_row["Model"])

Final selected model:
XGBoost


In [24]:
# Create models directory
os.makedirs("../models", exist_ok=True)

# ---------------------------------
# Select final classification model
# ---------------------------------

if tuned_accuracy >= best_classification_row["Accuracy"]:
    final_classification_pipeline = tuned_xgb
    final_classification_name = "Tuned XGBoost"
else:
    final_classification_pipeline = best_classification_pipeline
    final_classification_name = best_classification_name


# ---------------------------------
# Select final regression model
# ---------------------------------

final_regression_pipeline = regression_pipelines[
    best_regression_name
]


# ---------------------------------
# Save complete pipelines
# ---------------------------------

joblib.dump(
    final_classification_pipeline,
    "../models/classification_pipeline.pkl"
)

joblib.dump(
    final_regression_pipeline,
    "../models/regression_pipeline.pkl"
)

print("Models saved successfully!")
print()
print("Classification model:")
print(final_classification_name)

print("\nRegression model:")
print(best_regression_name)

Models saved successfully!

Classification model:
XGBoost

Regression model:
Random Forest


In [25]:
print("Classification pipeline exists:",
      os.path.exists("../models/classification_pipeline.pkl"))

print("Regression pipeline exists:",
      os.path.exists("../models/regression_pipeline.pkl"))

Classification pipeline exists: True
Regression pipeline exists: True


In [26]:
loaded_cls_pipeline = joblib.load(
    "../models/classification_pipeline.pkl"
)

saved_prediction = loaded_cls_pipeline.predict(X_test)

saved_accuracy = accuracy_score(
    y_cls_test,
    saved_prediction
)

print(
    f"Saved model test accuracy: {saved_accuracy * 100:.2f}%"
)

Saved model test accuracy: 89.87%


In [28]:
loaded_reg_pipeline = joblib.load(
    "../models/regression_pipeline.pkl"
)

saved_reg_prediction = loaded_reg_pipeline.predict(X_test)

saved_r2 = r2_score(
    y_reg_test,
    saved_reg_prediction
)

print(
    f"Saved regression model R²: {saved_r2:.4f}"
)

Saved regression model R²: 0.8661


In [29]:
print("=" * 50)
print("FINAL DATA LEAKAGE CHECK")
print("=" * 50)

print("\nInput features containing G3/result:")

leakage_features = [
    col for col in X.columns
    if "G3" in str(col).upper()
    or "RESULT" in str(col).upper()
]

if len(leakage_features) == 0:
    print("PASS: No direct target leakage found.")
else:
    print("WARNING:", leakage_features)

print("\nG3 used as feature:", "G3" in X.columns)
print("result used as feature:", "result" in X.columns)

FINAL DATA LEAKAGE CHECK

Input features containing G3/result:
PASS: No direct target leakage found.

G3 used as feature: False
result used as feature: False
